# Sentinel-1 Ship Detection: Final 3-Class Object Classification

This notebook converts existing object detections into the **three required final classes**:

1. **AIS_SUPPORTED_VESSEL**
2. **NON_AIS_SUPPORTED_VESSEL**
3. **STATIC_OR_PROBABLE_FALSE_ALARM**

The key design decision is that **`UNCERTAIN` is not a final class**. A detection without AIS is not automatically a false alarm, and it is not automatically a vessel. Instead, the notebook uses the Sentinel-1 **VV/VH multi-date signal**, spatial recurrence, local sea contrast, and object geometry to decide.

### Why this version is vessel-friendly without blindly converting uncertainty to vessels

Many visually plausible vessels can lack AIS. Therefore the non-AIS classifier is tuned with an **asymmetric objective**: retain high sensitivity to known AIS-supported vessels, then apply that learned vessel signature to unmatched detections. A high-confidence static rule is allowed to reject repeated fixed scatterers, but ordinary persistence alone is not enough to reject a target because anchored/moored vessels can recur.

### Important input assumption

This notebook starts from your **existing object-level detections** and **co-registered/georeferenced Sentinel-1 VV and VH rasters**. It does not rerun YOLO/CFAR detection from raw SAFE products. If the earlier notebook used different output names, update only the configuration cell below.

## Final decision logic

For each detection:

- **AIS-supported vessel**: retain any reliable AIS match.
- **Non-AIS-supported vessel**: no AIS match, not a hard land/static rejection, and either:
  - vessel score is above a threshold calibrated to preserve high recall on AIS-supported vessels, or
  - a conservative rescue rule finds strong local VV/VH contrast plus transient temporal behaviour and plausible geometry.
- **Static/probable false alarm**: on-land objects, highly fixed persistent scatterers with weak vessel evidence, or unmatched detections with vessel score below the calibrated threshold.

A `review_flag` is retained for QA, but it **does not create a fourth class**.

In [ ]:
# =============================
# 0. Imports
# =============================
import os, re, json, math, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from pyproj import Transformer, CRS

from sklearn.cluster import DBSCAN
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import recall_score, confusion_matrix, balanced_accuracy_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# =============================
# 1. USER CONFIGURATION
# Edit this cell first.
# =============================

# Search roots. Keep /kaggle/working first so final outputs from the old notebook are preferred.
PROJECT_ROOTS = [
    Path('/kaggle/working'),
    Path('/kaggle/input'),
]

# Optional: explicitly list object-level detection CSVs.
# Leave empty to auto-discover likely detection/object CSVs.
DETECTION_FILES = []

# Optional scene manifest with columns: scene_id, date, vv_path, vh_path
# This is the safest option if VV/VH filenames are unusual.
SCENE_MANIFEST_CSV = None

# Output
OUTPUT_DIR = Path('/kaggle/working/final_3class_vv_vh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raster interpretation: 'AUTO', 'DB', or 'LINEAR'.
RASTER_VALUES = 'AUTO'

# Spatial clustering of repeated objects across dates.
# 30 m is deliberately tighter than a normal AIS matching radius to avoid merging nearby ships.
CLUSTER_EPS_M = 30.0

# VV/VH sampling around each cluster centroid.
TARGET_RADIUS_M = 25.0
BG_INNER_RADIUS_M = 60.0
BG_OUTER_RADIUS_M = 120.0
MIN_VALID_PIXELS = 6

# AIS handling. If an explicit boolean AIS match already exists, it takes precedence.
AIS_MATCH_MAX_M = 500.0

# Time-series evidence.
# Per-scene bright threshold is learned from AIS-positive local contrast, with this floor.
MIN_VESSEL_CONTRAST_DB = 2.0
AIS_CONTRAST_QUANTILE = 0.10
STATIC_BRIGHT_PERSISTENCE = 0.82
STATIC_POSITION_STD_M = 20.0
STATIC_REL_MAD_MAX = 0.30

# Vessel-friendly score calibration.
# The model threshold is selected to target high recall on known AIS-supported vessels.
TARGET_AIS_RECALL = 0.95
MAX_ANCHOR_NEGATIVE_FPR = 0.20
FALLBACK_VESSEL_THRESHOLD = 0.45

# Rescue rule for obvious vessel-like unmatched targets.
RESCUE_MAX_BRIGHT_PERSISTENCE = 0.60
RESCUE_MIN_SPIKE_DB = 1.0

# Internal QA only. Final class still remains one of the required three.
REVIEW_MARGIN = 0.08

RANDOM_STATE = 42

print('Output directory:', OUTPUT_DIR)

In [ ]:
# =============================
# 2. Utility functions
# =============================
DATE_RE = re.compile(r'(20\d{6})')


def parse_date_token(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    s = str(value)
    m = DATE_RE.search(s.replace('-', '').replace('_', ''))
    if m:
        return m.group(1)
    try:
        return pd.to_datetime(s).strftime('%Y%m%d')
    except Exception:
        return None


def first_existing(df, names):
    low = {str(c).lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None


def to_bool_series(s):
    if s.dtype == bool:
        return s.fillna(False)
    vals = s.astype(str).str.strip().str.lower()
    return vals.isin(['1','true','t','yes','y','matched','ais','ais_matched','ais-supported'])


def safe_numeric(s):
    return pd.to_numeric(s, errors='coerce')


def robust_mad(x):
    a = np.asarray(x, dtype=float)
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return np.nan
    med = np.median(a)
    return np.median(np.abs(a - med))


def sigmoid(x):
    x = np.clip(np.asarray(x, dtype=float), -30, 30)
    return 1.0 / (1.0 + np.exp(-x))


def choose_utm_epsg(lon, lat):
    zone = int((float(lon) + 180) // 6) + 1
    return (32600 if float(lat) >= 0 else 32700) + zone

In [ ]:
# =============================
# 3. Discover / load the object-level detection tables
# =============================

def discover_detection_csvs(roots):
    candidates = []
    keywords = ('object', 'detect', 'classif', 'vessel', 'ship', 'final')
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob('*.csv'):
            name = p.name.lower()
            if any(k in name for k in keywords):
                candidates.append(p)
    # Working files first, then newer files.
    candidates = sorted(set(candidates), key=lambda p: ('/kaggle/working' not in str(p), -p.stat().st_mtime))
    return candidates


def normalize_detection_table(df, source_file):
    out = df.copy()

    mapping = {
        'scene_id': ['scene_id','scene','image_id','image','scene_name','sar_scene','scene_tag'],
        'date': ['date','scene_date','acquisition_date','acq_date','datetime','timestamp','acquisition_time'],
        'lon': ['lon','longitude','centroid_lon','center_lon','centre_lon','x_lon'],
        'lat': ['lat','latitude','centroid_lat','center_lat','centre_lat','y_lat'],
        'cx_px': ['cx','center_x','centre_x','x_center','xcentre','centroid_x_px','cx_px'],
        'cy_px': ['cy','center_y','centre_y','y_center','ycentre','centroid_y_px','cy_px'],
        'x1': ['x1','xmin','x_min','left'],
        'y1': ['y1','ymin','y_min','top'],
        'x2': ['x2','xmax','x_max','right'],
        'y2': ['y2','ymax','y_max','bottom'],
        'major_m': ['length_m','major_m','major_axis_m','bbox_length_m','target_length_m'],
        'minor_m': ['width_m','minor_m','minor_axis_m','bbox_width_m','target_width_m'],
        'area_m2': ['area_m2','target_area_m2','bbox_area_m2'],
        'aspect_ratio': ['aspect_ratio','aspect','elongation'],
        'confidence': ['confidence','conf','score','det_conf','yolo_confidence'],
        'ais_distance_m': ['ais_distance_m','ais_dist_m','distance_to_ais_m','nearest_ais_m','ais_match_distance_m'],
        'ais_matched': ['ais_matched','ais_match','matched_ais','is_ais_match','has_ais'],
        'on_land': ['on_land','is_land','land_flag','inside_land'],
        'water_fraction': ['water_fraction','water_frac','sea_fraction'],
        'distance_to_land_m': ['distance_to_land_m','dist_to_land_m','coast_distance_m','distance_to_coast_m'],
        'old_class': ['final_class','object_class','classification','category','class_name','label'],
    }

    for new, names in mapping.items():
        if new not in out.columns:
            c = first_existing(out, names)
            if c is not None:
                out[new] = out[c]

    # Infer scene/date from filename when needed.
    if 'scene_id' not in out.columns:
        out['scene_id'] = Path(source_file).stem
    out['scene_id'] = out['scene_id'].astype(str)

    if 'date' not in out.columns:
        out['date'] = out['scene_id'].map(parse_date_token)
    else:
        out['date'] = out['date'].map(parse_date_token)
        miss = out['date'].isna()
        out.loc[miss, 'date'] = out.loc[miss, 'scene_id'].map(parse_date_token)
    if out['date'].isna().all():
        file_date = parse_date_token(Path(source_file).name)
        if file_date:
            out['date'] = file_date

    for c in ['lon','lat','cx_px','cy_px','x1','y1','x2','y2','major_m','minor_m','area_m2',
              'aspect_ratio','confidence','ais_distance_m','water_fraction','distance_to_land_m']:
        if c in out.columns:
            out[c] = safe_numeric(out[c])

    if 'ais_matched' in out.columns:
        out['ais_matched'] = to_bool_series(out['ais_matched'])
    else:
        out['ais_matched'] = False

    if 'ais_distance_m' in out.columns:
        out['ais_matched'] = out['ais_matched'] | (out['ais_distance_m'] <= AIS_MATCH_MAX_M)

    if 'old_class' in out.columns:
        cls = out['old_class'].astype(str).str.upper().str.replace(' ', '_', regex=False)
        out['old_class'] = cls
        out['ais_matched'] = out['ais_matched'] | cls.str.contains('AIS_MATCHED|AIS_SUPPORTED', regex=True, na=False)
    else:
        out['old_class'] = ''

    if 'on_land' in out.columns:
        out['on_land'] = to_bool_series(out['on_land'])
    else:
        out['on_land'] = False

    if 'water_fraction' not in out.columns:
        out['water_fraction'] = np.where(out['on_land'], 0.0, np.nan)

    out['_source_csv'] = str(source_file)
    return out


if DETECTION_FILES:
    det_files = [Path(p) for p in DETECTION_FILES]
else:
    det_files = discover_detection_csvs(PROJECT_ROOTS)

print(f'Candidate detection CSVs found: {len(det_files)}')
for p in det_files[:25]:
    print(' ', p)

loaded = []
for p in det_files:
    try:
        d = pd.read_csv(p)
        nd = normalize_detection_table(d, p)
        # Keep only likely object tables.
        has_geo = ('lon' in nd.columns and 'lat' in nd.columns and nd[['lon','lat']].notna().any().all())
        has_px = ('cx_px' in nd.columns and 'cy_px' in nd.columns) or all(c in nd.columns for c in ['x1','y1','x2','y2'])
        has_scene = nd['date'].notna().any()
        if len(nd) > 0 and has_scene and (has_geo or has_px):
            loaded.append(nd)
    except Exception:
        pass

if not loaded:
    raise RuntimeError(
        'No usable object detection table was found. Set DETECTION_FILES explicitly to the CSV(s) produced by your earlier detection notebook.'
    )

det = pd.concat(loaded, ignore_index=True, sort=False)
# Remove exact duplicates that can happen if summary copies are discovered.
dedup_cols = [c for c in ['scene_id','date','lon','lat','cx_px','cy_px','x1','y1','x2','y2','confidence'] if c in det.columns]
if dedup_cols:
    det = det.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

det['object_id'] = np.arange(len(det), dtype=int)
print('\nLoaded objects:', len(det))
print('Scenes/dates:', det['date'].nunique())
print('AIS matches:', int(det['ais_matched'].sum()))
print('Columns:', list(det.columns))
display(det.head())

In [ ]:
# =============================
# 4. Build VV/VH scene manifest
# =============================

def pol_token(path, pol):
    s = path.stem.upper()
    return bool(re.search(rf'(^|[_\-.]){pol}([_\-.]|$)', s)) or (pol in s and ('SIGMA' in s or 'BACKSCATTER' in s or 'S1' in s))


def discover_rasters(roots):
    rasters=[]
    for root in roots:
        if not root.exists():
            continue
        rasters.extend(root.rglob('*.tif'))
        rasters.extend(root.rglob('*.tiff'))
    return sorted(set(rasters))


def build_manifest_from_files(roots):
    rasters = discover_rasters(roots)
    rows=[]
    by_date=defaultdict(lambda: {'VV':[], 'VH':[]})
    for p in rasters:
        dt = parse_date_token(p.name)
        if not dt:
            continue
        if pol_token(p, 'VV'):
            by_date[dt]['VV'].append(p)
        if pol_token(p, 'VH'):
            by_date[dt]['VH'].append(p)
    for dt, d in sorted(by_date.items()):
        if not d['VV'] or not d['VH']:
            continue
        # Prefer /kaggle/working and shortest path/name.
        rank = lambda p: ('/kaggle/working' not in str(p), len(str(p)))
        vv = sorted(d['VV'], key=rank)[0]
        vh = sorted(d['VH'], key=rank)[0]
        rows.append({'scene_id': dt, 'date': dt, 'vv_path': str(vv), 'vh_path': str(vh)})
    return pd.DataFrame(rows)


if SCENE_MANIFEST_CSV:
    manifest = pd.read_csv(SCENE_MANIFEST_CSV)
    need = {'scene_id','date','vv_path','vh_path'}
    missing = need - set(manifest.columns)
    if missing:
        raise ValueError(f'Manifest missing columns: {missing}')
    manifest['date'] = manifest['date'].map(parse_date_token)
else:
    manifest = build_manifest_from_files(PROJECT_ROOTS)

if manifest.empty:
    raise RuntimeError(
        'No paired VV/VH GeoTIFFs were found. Provide SCENE_MANIFEST_CSV with scene_id,date,vv_path,vh_path.'
    )

manifest = manifest.dropna(subset=['date']).drop_duplicates('date').sort_values('date').reset_index(drop=True)
print('VV/VH scene pairs:', len(manifest))
display(manifest)

# Restrict detections to dates with VV/VH data.
available_dates = set(manifest['date'])
missing_dates = sorted(set(det['date'].dropna()) - available_dates)
if missing_dates:
    print('WARNING: detection dates without VV/VH pair:', missing_dates)

det = det[det['date'].isin(available_dates)].copy().reset_index(drop=True)
if det.empty:
    raise RuntimeError('After matching dates, no detections remain. Check scene-date naming or use SCENE_MANIFEST_CSV.')

In [ ]:
# =============================
# 5. Fill lon/lat from pixel centers when necessary
# =============================

def pixel_center_from_bbox(df):
    if 'cx_px' not in df.columns and all(c in df.columns for c in ['x1','x2']):
        df['cx_px'] = (df['x1'] + df['x2']) / 2.0
    if 'cy_px' not in df.columns and all(c in df.columns for c in ['y1','y2']):
        df['cy_px'] = (df['y1'] + df['y2']) / 2.0
    return df


def fill_geolocation_from_rasters(det, manifest):
    det = pixel_center_from_bbox(det.copy())
    if 'lon' not in det.columns:
        det['lon'] = np.nan
    if 'lat' not in det.columns:
        det['lat'] = np.nan

    man = manifest.set_index('date')
    for dt, idx in det[det['lon'].isna() | det['lat'].isna()].groupby('date').groups.items():
        if dt not in man.index:
            continue
        if 'cx_px' not in det.columns or 'cy_px' not in det.columns:
            continue
        with rasterio.open(man.loc[dt, 'vv_path']) as src:
            rows = det.loc[idx, 'cy_px'].to_numpy(float)
            cols = det.loc[idx, 'cx_px'].to_numpy(float)
            xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset='center')
            if src.crs is None:
                continue
            lon, lat = rio_transform(src.crs, 'EPSG:4326', list(xs), list(ys))
            det.loc[idx, 'lon'] = lon
            det.loc[idx, 'lat'] = lat
    return det


det = fill_geolocation_from_rasters(det, manifest)
if det[['lon','lat']].isna().any().any():
    bad = det[det['lon'].isna() | det['lat'].isna()]
    raise RuntimeError(
        f'{len(bad)} detections still lack lon/lat. Provide geographic coordinates in the object CSV or pixel centers compatible with the VV raster.'
    )

print('Geolocation ready for', len(det), 'objects')

In [ ]:
# =============================
# 6. Cluster repeated detections across dates
# =============================
med_lon = float(det['lon'].median())
med_lat = float(det['lat'].median())
utm_epsg = choose_utm_epsg(med_lon, med_lat)
ll_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{utm_epsg}', always_xy=True)

det['x_m'], det['y_m'] = ll_to_utm.transform(det['lon'].to_numpy(), det['lat'].to_numpy())
coords = det[['x_m','y_m']].to_numpy(float)
labels = DBSCAN(eps=CLUSTER_EPS_M, min_samples=1, metric='euclidean').fit_predict(coords)
det['temporal_cluster_id'] = labels.astype(int)

n_scenes_total = manifest['date'].nunique()
cluster_base = det.groupby('temporal_cluster_id').agg(
    cluster_lon=('lon','median'),
    cluster_lat=('lat','median'),
    n_detection_rows=('object_id','size'),
    n_dates_detected=('date','nunique'),
    x_mean=('x_m','mean'), y_mean=('y_m','mean'),
    x_std=('x_m','std'), y_std=('y_m','std'),
).reset_index()
cluster_base['position_std_m'] = np.sqrt(cluster_base['x_std'].fillna(0)**2 + cluster_base['y_std'].fillna(0)**2)
cluster_base['detection_persistence'] = cluster_base['n_dates_detected'] / max(n_scenes_total, 1)

print('Spatial-temporal clusters:', len(cluster_base))
print(cluster_base[['n_dates_detected','detection_persistence','position_std_m']].describe())

In [ ]:
# =============================
# 7. VV/VH sampling functions
# =============================

def infer_db_mode(arr):
    vals = arr[np.isfinite(arr)]
    if len(vals) == 0:
        return 'DB'
    q05, q50, q95 = np.nanpercentile(vals, [5,50,95])
    # Typical sigma0 linear is non-negative and often below ~2; dB generally contains negatives.
    if q05 >= 0 and q95 <= 5:
        return 'LINEAR'
    return 'DB'


def convert_to_db(arr, mode='AUTO'):
    a = np.asarray(arr, dtype='float32')
    m = mode.upper()
    if m == 'AUTO':
        m = infer_db_mode(a)
    if m == 'LINEAR':
        out = np.full_like(a, np.nan, dtype='float32')
        good = np.isfinite(a) & (a > 0)
        out[good] = 10.0 * np.log10(a[good])
        return out
    return a


def meters_per_pixel(src, lat):
    rx, ry = abs(src.transform.a), abs(src.transform.e)
    if src.crs and src.crs.is_geographic:
        x_m = rx * 111320.0 * max(math.cos(math.radians(float(lat))), 0.05)
        y_m = ry * 110540.0
    else:
        # Most projected CRS used for S1 rasters are in metres.
        x_m, y_m = rx, ry
    return max(float(x_m), 0.1), max(float(y_m), 0.1)


def sample_pol_window(src, lon, lat, target_radius_m, bg_inner_m, bg_outer_m):
    if src.crs is None:
        return None
    xs, ys = rio_transform('EPSG:4326', src.crs, [float(lon)], [float(lat)])
    x, y = xs[0], ys[0]
    row, col = src.index(x, y)
    if row < 0 or col < 0 or row >= src.height or col >= src.width:
        return None

    x_mpp, y_mpp = meters_per_pixel(src, lat)
    half_cols = int(math.ceil(bg_outer_m / x_mpp)) + 2
    half_rows = int(math.ceil(bg_outer_m / y_mpp)) + 2
    r0, c0 = row-half_rows, col-half_cols
    h, w = 2*half_rows+1, 2*half_cols+1
    win = Window(c0, r0, w, h)
    arr = src.read(1, window=win, boundless=True, fill_value=np.nan).astype('float32')
    if src.nodata is not None:
        arr[arr == src.nodata] = np.nan
    arr = convert_to_db(arr, RASTER_VALUES)

    yy = (np.arange(arr.shape[0]) - half_rows) * y_mpp
    xx = (np.arange(arr.shape[1]) - half_cols) * x_mpp
    XX, YY = np.meshgrid(xx, yy)
    rr = np.sqrt(XX**2 + YY**2)
    target = arr[(rr <= target_radius_m) & np.isfinite(arr)]
    bg = arr[(rr >= bg_inner_m) & (rr <= bg_outer_m) & np.isfinite(arr)]
    if len(target) < MIN_VALID_PIXELS or len(bg) < MIN_VALID_PIXELS:
        return None

    bg_med = float(np.nanmedian(bg))
    target_med = float(np.nanmedian(target))
    target_p95 = float(np.nanpercentile(target, 95))
    return {
        'target_median_db': target_med,
        'target_p95_db': target_p95,
        'background_median_db': bg_med,
        'contrast_db': target_p95 - bg_med,
        'bright_fraction_3db': float(np.mean(target >= (bg_med + 3.0))),
        'n_target_px': int(len(target)),
        'n_bg_px': int(len(bg)),
    }


def sample_cluster_timeseries(cluster_base, manifest):
    rows=[]
    clusters = cluster_base[['temporal_cluster_id','cluster_lon','cluster_lat']].to_dict('records')
    for i, sc in manifest.iterrows():
        print(f"Sampling {i+1}/{len(manifest)}: {sc['date']}")
        with rasterio.open(sc['vv_path']) as vv_src, rasterio.open(sc['vh_path']) as vh_src:
            for c in clusters:
                vv = sample_pol_window(vv_src, c['cluster_lon'], c['cluster_lat'], TARGET_RADIUS_M, BG_INNER_RADIUS_M, BG_OUTER_RADIUS_M)
                vh = sample_pol_window(vh_src, c['cluster_lon'], c['cluster_lat'], TARGET_RADIUS_M, BG_INNER_RADIUS_M, BG_OUTER_RADIUS_M)
                if vv is None and vh is None:
                    continue
                rec = {'temporal_cluster_id': c['temporal_cluster_id'], 'date': sc['date']}
                if vv is not None:
                    rec.update({f'vv_{k}': v for k,v in vv.items()})
                if vh is not None:
                    rec.update({f'vh_{k}': v for k,v in vh.items()})
                rows.append(rec)
    return pd.DataFrame(rows)


ts = sample_cluster_timeseries(cluster_base, manifest)
if ts.empty:
    raise RuntimeError('No VV/VH samples could be extracted. Check raster CRS, extent and detection coordinates.')

for c in ['vv_contrast_db','vh_contrast_db']:
    if c not in ts.columns:
        ts[c] = np.nan

ts['combined_contrast_db'] = ts[['vv_contrast_db','vh_contrast_db']].max(axis=1, skipna=True)
ts['vv_minus_vh_peak_db'] = ts.get('vv_target_p95_db', np.nan) - ts.get('vh_target_p95_db', np.nan)
print('Time-series rows:', len(ts))
display(ts.head())

In [ ]:
# =============================
# 8. Attach current-scene VV/VH evidence to each object
# =============================
cur_cols = [
    'temporal_cluster_id','date',
    'vv_target_p95_db','vh_target_p95_db',
    'vv_background_median_db','vh_background_median_db',
    'vv_contrast_db','vh_contrast_db','combined_contrast_db',
    'vv_minus_vh_peak_db','vv_bright_fraction_3db','vh_bright_fraction_3db'
]
cur_cols = [c for c in cur_cols if c in ts.columns]
det = det.merge(ts[cur_cols], on=['temporal_cluster_id','date'], how='left')

# Add cluster recurrence basics.
det = det.merge(cluster_base[[
    'temporal_cluster_id','n_dates_detected','detection_persistence','position_std_m'
]], on='temporal_cluster_id', how='left')

# Current temporal spike relative to the SAME location on all other dates.
med_all = ts.groupby('temporal_cluster_id')['combined_contrast_db'].median().rename('ts_median_contrast_db')
mad_all = ts.groupby('temporal_cluster_id')['combined_contrast_db'].apply(robust_mad).rename('ts_mad_contrast_db')
max_all = ts.groupby('temporal_cluster_id')['combined_contrast_db'].max().rename('ts_max_contrast_db')
valid_all = ts.groupby('temporal_cluster_id')['combined_contrast_db'].count().rename('n_valid_ts_dates')
cluster_ts_summary = pd.concat([med_all,mad_all,max_all,valid_all], axis=1).reset_index()
cluster_ts_summary['ts_relative_mad'] = cluster_ts_summary['ts_mad_contrast_db'] / (cluster_ts_summary['ts_median_contrast_db'].abs() + 1.0)

det = det.merge(cluster_ts_summary, on='temporal_cluster_id', how='left')
det['temporal_spike_db'] = det['combined_contrast_db'] - det['ts_median_contrast_db']

In [ ]:
# =============================
# 9. Learn a per-scene "bright vessel" contrast threshold from AIS positives
# =============================
# Known AIS matches act as positive reference signatures only.
ais_ref = det[det['ais_matched'] & det['combined_contrast_db'].notna()].copy()
if len(ais_ref) < 5:
    raise RuntimeError('Too few AIS-supported objects with VV/VH features to calibrate vessel evidence.')

global_ais_thr = max(MIN_VESSEL_CONTRAST_DB, float(ais_ref['combined_contrast_db'].quantile(AIS_CONTRAST_QUANTILE)))
scene_thr = {}
for dt, g in ais_ref.groupby('date'):
    if len(g) >= 5:
        scene_thr[dt] = max(MIN_VESSEL_CONTRAST_DB, float(g['combined_contrast_db'].quantile(AIS_CONTRAST_QUANTILE)))
    else:
        scene_thr[dt] = global_ais_thr

manifest['vessel_contrast_thr_db'] = manifest['date'].map(scene_thr).fillna(global_ais_thr)
print('Global AIS-derived contrast floor:', round(global_ais_thr, 2), 'dB')
display(manifest[['date','vessel_contrast_thr_db']])

thr_map = dict(zip(manifest['date'], manifest['vessel_contrast_thr_db']))
ts['scene_contrast_thr_db'] = ts['date'].map(thr_map).fillna(global_ais_thr)
ts['bright_like_vessel'] = ts['combined_contrast_db'] >= ts['scene_contrast_thr_db']

bright_summary = ts.groupby('temporal_cluster_id').agg(
    n_valid_ts_dates2=('combined_contrast_db','count'),
    n_bright_dates=('bright_like_vessel','sum'),
).reset_index()
bright_summary['bright_persistence'] = bright_summary['n_bright_dates'] / bright_summary['n_valid_ts_dates2'].clip(lower=1)

det = det.merge(bright_summary[['temporal_cluster_id','bright_persistence']], on='temporal_cluster_id', how='left')
det['scene_contrast_thr_db'] = det['date'].map(thr_map).fillna(global_ais_thr)
det['current_bright_like_vessel'] = det['combined_contrast_db'] >= det['scene_contrast_thr_db']

print(det[['combined_contrast_db','scene_contrast_thr_db','bright_persistence','temporal_spike_db']].describe())

In [ ]:
# =============================
# 10. Geometry normalization and broad physical plausibility
# =============================

def raster_mpp_for_date(date, lon, lat):
    r = manifest[manifest['date'] == date]
    if r.empty:
        return np.nan, np.nan
    try:
        with rasterio.open(r.iloc[0]['vv_path']) as src:
            return meters_per_pixel(src, lat)
    except Exception:
        return np.nan, np.nan

# Derive pixel bbox dimensions if available.
if all(c in det.columns for c in ['x1','x2','y1','y2']):
    det['bbox_w_px'] = (det['x2'] - det['x1']).abs()
    det['bbox_h_px'] = (det['y2'] - det['y1']).abs()

if 'major_m' not in det.columns:
    det['major_m'] = np.nan
if 'minor_m' not in det.columns:
    det['minor_m'] = np.nan

need_geom = det['major_m'].isna() | det['minor_m'].isna()
if need_geom.any() and 'bbox_w_px' in det.columns:
    cache_mpp={}
    for idx in det.index[need_geom]:
        key=(det.at[idx,'date'], round(det.at[idx,'lat'],4))
        if key not in cache_mpp:
            cache_mpp[key]=raster_mpp_for_date(det.at[idx,'date'], det.at[idx,'lon'], det.at[idx,'lat'])
        xm, ym = cache_mpp[key]
        w = det.at[idx,'bbox_w_px'] * xm if np.isfinite(xm) else np.nan
        h = det.at[idx,'bbox_h_px'] * ym if np.isfinite(ym) else np.nan
        det.at[idx,'major_m'] = np.nanmax([w,h]) if np.isfinite(w) or np.isfinite(h) else np.nan
        det.at[idx,'minor_m'] = np.nanmin([w,h]) if np.isfinite(w) and np.isfinite(h) else np.nan

if 'aspect_ratio' not in det.columns:
    det['aspect_ratio'] = det['major_m'] / det['minor_m'].replace(0, np.nan)
if 'area_m2' not in det.columns:
    det['area_m2'] = det['major_m'] * det['minor_m']

# Do NOT make geometry too restrictive. Non-AIS vessels can be small.
geom_known = det['major_m'].notna() & det['minor_m'].notna()
det['geometry_plausible'] = True
det.loc[geom_known, 'geometry_plausible'] = (
    det.loc[geom_known, 'major_m'].between(4, 500) &
    det.loc[geom_known, 'minor_m'].between(1.5, 180) &
    det.loc[geom_known, 'aspect_ratio'].between(1.0, 25.0) &
    (det.loc[geom_known, 'area_m2'] <= 60000)
)

print('Geometry available:', int(geom_known.sum()), '/', len(det))
print('Broadly plausible geometry:', int(det['geometry_plausible'].sum()), '/', len(det))

In [ ]:
# =============================
# 11. Build high-confidence proxy anchors
# =============================
# Positive anchors are reliable AIS-supported vessels.
pos_anchor = det['ais_matched']

# Strong static signature: repeatedly bright at virtually the same location with low temporal variability.
strong_static_signature = (
    (det['bright_persistence'] >= STATIC_BRIGHT_PERSISTENCE) &
    (det['position_std_m'] <= STATIC_POSITION_STD_M) &
    (det['ts_relative_mad'] <= STATIC_REL_MAD_MAX) &
    (det['n_valid_ts_dates'] >= 4)
)

old_probable_false = det['old_class'].astype(str).str.contains('PROBABLE_FALSE|FALSE_ALARM', regex=True, na=False)
old_static = det['old_class'].astype(str).str.contains('STATIC', regex=True, na=False)

# Negative anchors are deliberately conservative.
# We do NOT train on all old UNCERTAIN or all old STATIC objects.
neg_anchor = (
    det['on_land'] |
    (old_probable_false & ~det['current_bright_like_vessel']) |
    (old_static & strong_static_signature & ~det['geometry_plausible']) |
    (strong_static_signature & ~det['geometry_plausible'])
)
neg_anchor = neg_anchor & ~pos_anchor

det['positive_anchor'] = pos_anchor
det['negative_anchor'] = neg_anchor

print('Positive AIS anchors:', int(pos_anchor.sum()))
print('High-confidence negative anchors:', int(neg_anchor.sum()))
print('Strong static signatures:', int(strong_static_signature.sum()))

In [ ]:
# =============================
# 12. Train interpretable vessel-evidence model
# =============================
# The model is not trained on "uncertain" labels. It learns the feature difference between
# known AIS vessels and conservative static/false anchors, then scores all unmatched detections.

feature_cols = [
    'vv_contrast_db','vh_contrast_db','combined_contrast_db','vv_minus_vh_peak_db',
    'temporal_spike_db','bright_persistence','detection_persistence','position_std_m',
    'ts_relative_mad','major_m','minor_m','area_m2','aspect_ratio',
    'confidence','water_fraction','distance_to_land_m'
]
feature_cols = [c for c in feature_cols if c in det.columns and det[c].notna().any()]

anchor_mask = det['positive_anchor'] | det['negative_anchor']
X_anchor = det.loc[anchor_mask, feature_cols].copy()
y_anchor = det.loc[anchor_mask, 'positive_anchor'].astype(int).to_numpy()

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=3000, C=1.0, random_state=RANDOM_STATE)),
])

use_model = (np.sum(y_anchor==1) >= 20 and np.sum(y_anchor==0) >= 20 and len(feature_cols) >= 4)
calibrated_threshold = FALLBACK_VESSEL_THRESHOLD
cv_info = {}

if use_model:
    min_class = int(min(np.sum(y_anchor==1), np.sum(y_anchor==0)))
    n_splits = max(2, min(5, min_class))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = cross_val_predict(model, X_anchor, y_anchor, cv=cv, method='predict_proba')[:,1]

    candidates = np.linspace(0.05, 0.95, 181)
    rows=[]
    for t in candidates:
        pred = (oof >= t).astype(int)
        rec = recall_score(y_anchor, pred, zero_division=0)
        cm = confusion_matrix(y_anchor, pred, labels=[0,1])
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / max(fp+tn, 1)
        bal = balanced_accuracy_score(y_anchor, pred)
        rows.append((t, rec, fpr, bal))
    thr_df = pd.DataFrame(rows, columns=['threshold','ais_recall','negative_fpr','balanced_accuracy'])

    feasible = thr_df[(thr_df['ais_recall'] >= TARGET_AIS_RECALL) & (thr_df['negative_fpr'] <= MAX_ANCHOR_NEGATIVE_FPR)]
    if not feasible.empty:
        # Highest threshold satisfying the high-recall requirement gives the best specificity.
        calibrated_threshold = float(feasible.sort_values(['threshold','balanced_accuracy'], ascending=[False,False]).iloc[0]['threshold'])
    else:
        high_recall = thr_df[thr_df['ais_recall'] >= TARGET_AIS_RECALL]
        if not high_recall.empty:
            calibrated_threshold = float(high_recall['threshold'].max())
        else:
            calibrated_threshold = float(thr_df.sort_values('balanced_accuracy', ascending=False).iloc[0]['threshold'])

    model.fit(X_anchor, y_anchor)
    det['vessel_score'] = model.predict_proba(det[feature_cols])[:,1]

    pred_oof = (oof >= calibrated_threshold).astype(int)
    cm = confusion_matrix(y_anchor, pred_oof, labels=[0,1])
    cv_info = {
        'threshold': calibrated_threshold,
        'oof_ais_recall': float(recall_score(y_anchor, pred_oof, zero_division=0)),
        'oof_balanced_accuracy': float(balanced_accuracy_score(y_anchor, pred_oof)),
        'oof_confusion_matrix': cm.tolist(),
        'n_positive_anchors': int(np.sum(y_anchor==1)),
        'n_negative_anchors': int(np.sum(y_anchor==0)),
        'features': feature_cols,
    }
    print('Model-based vessel score enabled')
    print(json.dumps(cv_info, indent=2))
else:
    # Transparent fallback score if there are too few conservative negative anchors.
    sar = sigmoid((det['combined_contrast_db'] - det['scene_contrast_thr_db']) / 1.5)
    transient = (1.0 - det['bright_persistence'].fillna(0.5)).clip(0,1)
    spike = sigmoid(det['temporal_spike_db'].fillna(0.0) / 1.5)
    geom = det['geometry_plausible'].astype(float)
    static_pen = strong_static_signature.astype(float)
    land_pen = det['on_land'].astype(float)
    det['vessel_score'] = (0.48*sar + 0.24*transient + 0.18*spike + 0.10*geom - 0.30*static_pen - 0.50*land_pen).clip(0,1)
    calibrated_threshold = FALLBACK_VESSEL_THRESHOLD
    cv_info = {'mode':'fallback_rule_score', 'threshold': calibrated_threshold}
    print('WARNING: too few conservative anchors. Using transparent fallback vessel score.')

print('Final vessel-score threshold:', round(calibrated_threshold, 3))
print(det['vessel_score'].describe())

In [ ]:
# =============================
# 13. FINAL THREE-CLASS DECISION
# =============================
# Important: persistence alone is NOT a hard rejection because moored/anchored vessels can recur.
# A static hard override requires a strong fixed-scatterer signature AND insufficient vessel evidence.

det['hard_static_signature'] = strong_static_signature

# Rescue clear vessel-like detections that a proxy-label model may score too low.
# This is particularly important for small non-AIS vessels not represented by AIS positives.
rescue_vessel = (
    (~det['ais_matched']) &
    (~det['on_land']) &
    det['geometry_plausible'] &
    det['current_bright_like_vessel'] &
    (
        (det['bright_persistence'] <= RESCUE_MAX_BRIGHT_PERSISTENCE) |
        (det['temporal_spike_db'] >= RESCUE_MIN_SPIKE_DB)
    )
)

# If a target looks highly static, only force it to class 3 when the vessel score is not strongly positive.
hard_static_reject = (
    det['hard_static_signature'] &
    (det['vessel_score'] < max(calibrated_threshold + 0.15, 0.65)) &
    (~det['ais_matched'])
)

final = np.full(len(det), 'STATIC_OR_PROBABLE_FALSE_ALARM', dtype=object)
final[det['ais_matched'].to_numpy()] = 'AIS_SUPPORTED_VESSEL'

non_ais_vessel = (
    (~det['ais_matched']) &
    (~det['on_land']) &
    (~hard_static_reject) &
    (
        (det['vessel_score'] >= calibrated_threshold) |
        rescue_vessel
    )
)
final[non_ais_vessel.to_numpy()] = 'NON_AIS_SUPPORTED_VESSEL'
det['final_class_3'] = final

# Explainable reason field.
reason=[]
for i,r in det.iterrows():
    if r['ais_matched']:
        reason.append('AIS match')
    elif bool(r['on_land']):
        reason.append('on-land hard rejection')
    elif bool(hard_static_reject.loc[i]):
        reason.append('high fixed temporal persistence + low vessel evidence')
    elif bool(rescue_vessel.loc[i]) and r['vessel_score'] < calibrated_threshold:
        reason.append('VV/VH vessel rescue: strong current contrast + transient/spike evidence')
    elif r['vessel_score'] >= calibrated_threshold:
        reason.append(f'vessel score >= calibrated threshold ({calibrated_threshold:.2f})')
    else:
        reason.append(f'vessel score below calibrated threshold ({calibrated_threshold:.2f})')
det['classification_reason'] = reason

# QA flag only, not a class.
det['review_flag'] = (
    ((det['vessel_score'] - calibrated_threshold).abs() <= REVIEW_MARGIN) |
    (hard_static_reject & (det['vessel_score'] >= calibrated_threshold - 0.10)) |
    (rescue_vessel & (det['vessel_score'] < calibrated_threshold))
) & (~det['ais_matched'])

print('\nFINAL CLASS COUNTS')
print(det['final_class_3'].value_counts())
print('\nQA review flags:', int(det['review_flag'].sum()))
assert set(det['final_class_3'].unique()).issubset({
    'AIS_SUPPORTED_VESSEL','NON_AIS_SUPPORTED_VESSEL','STATIC_OR_PROBABLE_FALSE_ALARM'
})

In [ ]:
# =============================
# 14. Inspect how old UNCERTAIN detections were redistributed
# =============================
old_uncertain = det['old_class'].astype(str).str.contains('UNCERTAIN', na=False)
print('Old UNCERTAIN objects in matched VV/VH scenes:', int(old_uncertain.sum()))
if old_uncertain.any():
    print('\nRedistribution of old UNCERTAIN:')
    print(det.loc[old_uncertain, 'final_class_3'].value_counts())
    print('\nVessel-score summary for old UNCERTAIN:')
    display(det.loc[old_uncertain, [
        'vessel_score','combined_contrast_db','temporal_spike_db','bright_persistence',
        'position_std_m','geometry_plausible','final_class_3','classification_reason'
    ]].describe(include='all'))

# Old-to-new matrix
matrix = pd.crosstab(det['old_class'].replace('', 'NO_OLD_CLASS'), det['final_class_3'])
display(matrix)

In [ ]:
# =============================
# 15. Diagnostics: score distributions
# =============================
fig, ax = plt.subplots(figsize=(10,5))
for label, mask in [
    ('AIS supported', det['ais_matched']),
    ('Old uncertain', old_uncertain & ~det['ais_matched']),
    ('Old probable false', old_probable_false & ~det['ais_matched']),
]:
    vals = det.loc[mask, 'vessel_score'].dropna()
    if len(vals):
        ax.hist(vals, bins=30, alpha=0.45, label=f'{label} (n={len(vals)})')
ax.axvline(calibrated_threshold, linestyle='--', linewidth=2, label=f'final threshold={calibrated_threshold:.2f}')
ax.set_xlabel('Vessel evidence score')
ax.set_ylabel('Objects')
ax.set_title('Vessel-score diagnostic')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'vessel_score_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================
# 16. Per-scene final counts
# =============================
scene_counts = pd.crosstab(det['date'], det['final_class_3']).reset_index()
display(scene_counts)
scene_counts.to_csv(OUTPUT_DIR/'final_3class_scene_counts.csv', index=False)

In [ ]:
# =============================
# 17. Save object-level features and final classifications
# =============================
# Compact final table + full audit table.
keep_cols = [c for c in [
    'object_id','scene_id','date','lon','lat','old_class',
    'ais_matched','ais_distance_m','on_land','water_fraction','distance_to_land_m',
    'major_m','minor_m','area_m2','aspect_ratio','confidence',
    'vv_target_p95_db','vh_target_p95_db','vv_background_median_db','vh_background_median_db',
    'vv_contrast_db','vh_contrast_db','combined_contrast_db','vv_minus_vh_peak_db',
    'temporal_cluster_id','n_dates_detected','detection_persistence','bright_persistence',
    'position_std_m','ts_median_contrast_db','ts_mad_contrast_db','ts_relative_mad','temporal_spike_db',
    'geometry_plausible','vessel_score','final_class_3','classification_reason','review_flag'
] if c in det.columns]

final_table = det[keep_cols].copy()
final_table.to_csv(OUTPUT_DIR/'FINAL_3CLASS_OBJECTS.csv', index=False)
det.to_csv(OUTPUT_DIR/'FINAL_3CLASS_OBJECTS_FULL_AUDIT.csv', index=False)
ts.to_csv(OUTPUT_DIR/'CLUSTER_VV_VH_TIME_SERIES.csv', index=False)
manifest.to_csv(OUTPUT_DIR/'SCENE_MANIFEST_USED.csv', index=False)

with open(OUTPUT_DIR/'CLASSIFICATION_CONFIG_AND_CV.json','w') as f:
    json.dump({
        'cluster_eps_m': CLUSTER_EPS_M,
        'target_radius_m': TARGET_RADIUS_M,
        'bg_inner_radius_m': BG_INNER_RADIUS_M,
        'bg_outer_radius_m': BG_OUTER_RADIUS_M,
        'min_vessel_contrast_db': MIN_VESSEL_CONTRAST_DB,
        'ais_contrast_quantile': AIS_CONTRAST_QUANTILE,
        'static_bright_persistence': STATIC_BRIGHT_PERSISTENCE,
        'static_position_std_m': STATIC_POSITION_STD_M,
        'static_rel_mad_max': STATIC_REL_MAD_MAX,
        'target_ais_recall': TARGET_AIS_RECALL,
        'final_vessel_threshold': calibrated_threshold,
        'cv': cv_info,
    }, f, indent=2)

print('Saved:', OUTPUT_DIR/'FINAL_3CLASS_OBJECTS.csv')
display(final_table.head())

In [ ]:
# =============================
# 18. Final 3-class overlay for every scene
# =============================
CLASS_STYLE = {
    'AIS_SUPPORTED_VESSEL': ('limegreen', 'AIS-supported vessel'),
    'NON_AIS_SUPPORTED_VESSEL': ('deepskyblue', 'Non-AIS-supported vessel'),
    'STATIC_OR_PROBABLE_FALSE_ALARM': ('red', 'Static / probable false alarm'),
}


def raster_display_db(src, max_size=1800):
    scale = max(src.width/max_size, src.height/max_size, 1)
    out_h = max(1, int(src.height/scale))
    out_w = max(1, int(src.width/scale))
    arr = src.read(1, out_shape=(out_h,out_w)).astype('float32')
    if src.nodata is not None:
        arr[arr == src.nodata] = np.nan
    arr = convert_to_db(arr, RASTER_VALUES)
    b = src.bounds
    return arr, [b.left,b.right,b.bottom,b.top]


def plot_scene_overlay(date, scene_df, row):
    with rasterio.open(row['vv_path']) as src:
        bg, extent = raster_display_db(src)
        vmin, vmax = np.nanpercentile(bg[np.isfinite(bg)], [2, 98]) if np.isfinite(bg).any() else (-25, 5)
        fig, ax = plt.subplots(figsize=(18,10))
        ax.imshow(bg, cmap='gray', vmin=vmin, vmax=vmax, extent=extent, origin='upper')

        # Project lon/lat to raster CRS.
        xs, ys = rio_transform('EPSG:4326', src.crs, scene_df['lon'].tolist(), scene_df['lat'].tolist())
        tmp = scene_df.copy()
        tmp['_px'] = xs; tmp['_py'] = ys

        # Use a square marker in display coordinates. Size is intentionally modest to avoid hiding dense vessels.
        for cls, (color, label) in CLASS_STYLE.items():
            g = tmp[tmp['final_class_3'] == cls]
            if g.empty:
                continue
            ax.scatter(g['_px'], g['_py'], s=34, marker='s', facecolors='none', edgecolors=color,
                       linewidths=1.1, label=f'{label} ({len(g)})')

        ax.set_title(f'{date} - final 3-class object classification')
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.set_axis_off()
        ax.legend(loc='lower right', framealpha=0.9)
        plt.tight_layout()
        out = OUTPUT_DIR/f'{date}_FINAL_3CLASS_OVERLAY.png'
        plt.savefig(out, dpi=200, bbox_inches='tight')
        plt.close(fig)
        return out

for _, row in manifest.iterrows():
    g = det[det['date'] == row['date']]
    if g.empty:
        continue
    out = plot_scene_overlay(row['date'], g, row)
    print('Saved', out)

In [ ]:
# =============================
# 19. Final report summary
# =============================
summary = {
    'n_objects': int(len(det)),
    'n_scenes': int(det['date'].nunique()),
    'class_counts': det['final_class_3'].value_counts().to_dict(),
    'old_uncertain_count': int(old_uncertain.sum()),
    'old_uncertain_to_non_ais_vessel': int(((old_uncertain) & (det['final_class_3']=='NON_AIS_SUPPORTED_VESSEL')).sum()),
    'old_uncertain_to_static_false': int(((old_uncertain) & (det['final_class_3']=='STATIC_OR_PROBABLE_FALSE_ALARM')).sum()),
    'qa_review_flags': int(det['review_flag'].sum()),
    'vessel_score_threshold': float(calibrated_threshold),
}
print(json.dumps(summary, indent=2))
with open(OUTPUT_DIR/'FINAL_SUMMARY.json','w') as f:
    json.dump(summary, f, indent=2)

print('\nDone. Final class is ALWAYS exactly one of the three requested categories.')
print('Primary output:', OUTPUT_DIR/'FINAL_3CLASS_OBJECTS.csv')

## How to interpret the result

The most important table is `FINAL_3CLASS_OBJECTS.csv`.

For the detections that were previously `UNCERTAIN`, inspect:

- `combined_contrast_db`
- `temporal_spike_db`
- `bright_persistence`
- `position_std_m`
- `geometry_plausible`
- `vessel_score`
- `classification_reason`
- `review_flag`

A large fraction of old uncertain detections may legitimately move to **NON_AIS_SUPPORTED_VESSEL** if they resemble AIS-supported vessels in VV/VH contrast and are not strongly fixed persistent scatterers. This is intentional. What the notebook does **not** do is assume that every uncertain object is a vessel.

### Recommended acceptance check before reporting the result

1. Look at the old-uncertain redistribution table.
2. Check the vessel-score histogram. AIS-supported objects should mostly sit on the vessel side of the threshold.
3. Inspect the 3-class overlays around dense port/anchorage zones.
4. Review `review_flag=True` objects manually as QA, while keeping their forced three-class label for the required product.
5. If AIS recall in the printed cross-validation diagnostic is much below the requested target, do not trust the automatically calibrated threshold. Check the input VV/VH scaling and sampling first.